# OCCAM Reconstructability Analysis
## Canonical Notebook for PyOccam Package

This notebook provides a complete workflow for OCCAM analysis:
1. **Load** your data
2. **Search** for optimal models
3. **Fit** the best model with detailed statistics
4. **Visualize** results including confusion matrices

---

## 📋 Configuration
**Modify these settings for your analysis:**

In [1]:
# ============================================================================
# USER CONFIGURATION - MODIFY THESE SETTINGS
# ============================================================================

# Data file (required) - Path to your OCCAM data file
DATA_FILE = "dementia05.txt"  # <-- CHANGE THIS TO YOUR DATA FILE

# Search parameters
SEARCH_LEVELS = 3      # Number of levels to search (1-7, default: 3)
SEARCH_WIDTH = 3       # Width of search beam (1-10, default: 3)
SEARCH_TYPE = "loopless-up"  # Options: loopless-up, disjoint-up, chain-up, full-up

# Model selection criterion
SELECTION_CRITERION = "BIC"  # Options: "BIC", "AIC", "Information"

# Confusion matrix settings (for classification problems)
TARGET_STATE = "0"     # The 'negative' state for confusion matrix (e.g., "0" or "no")
                       # Leave empty "" to skip confusion matrix

# Output format for saved files
FILE_FORMAT = "space"  # Options: "tab", "comma", "space"

# Display options
SHOW_SEARCH_DETAILS = True  # Show detailed search results
SHOW_FIT_DETAILS = True     # Show detailed fit analysis

## 🚀 Initialize OCCAM

In [2]:
# Import required libraries
import pyoccam2 as pyoccam
from IPython.display import HTML, display
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Optional imports for enhanced features
try:
    import pandas as pd
    pd.set_option('display.max_rows', 20)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False
    print("Note: Install pandas for enhanced table display: pip install pandas")

# Helper function for beautiful HTML display in notebooks
def display_occam_html(content, title=""):
    """Display OCCAM output with enhanced formatting"""
    html = f"""
    <style>
        .occam-container {{
            font-family: 'Courier New', monospace;
            margin: 20px 0;
            background: white;
            border: 1px solid #e0e0e0;
            border-radius: 8px;
            padding: 15px;
        }}
        .occam-title {{
            font-size: 18px;
            font-weight: bold;
            color: #1a73e8;
            margin-bottom: 15px;
            padding: 10px;
            background: linear-gradient(90deg, #f0f7ff 0%, #ffffff 100%);
            border-left: 4px solid #1a73e8;
        }}
        .occam-container table {{
            border-collapse: collapse;
            margin: 10px 0;
            font-size: 12px;
        }}
        .occam-container td, .occam-container th {{
            padding: 5px 10px;
            text-align: right;
            border: 1px solid #ddd;
        }}
        .occam-container th {{
            background-color: #f5f5f5;
            font-weight: bold;
        }}
        .occam-container .r1 {{
            background-color: #f9f9f9;
        }}
        .occam-container .em {{
            font-weight: bold;
            background-color: #fff3cd;
        }}
        .occam-container pre {{
            background: #f8f8f8;
            padding: 10px;
            border-radius: 4px;
            overflow-x: auto;
        }}
    </style>
    <div class=\"occam-container\">
    """
    if title:
        html += f'<div class=\"occam-title\">{title}</div>'
    html += content + "</div>"
    display(HTML(html))

print("🧮 OCCAM Reconstructability Analysis System")
print("="*60)
print(f"Version: {pyoccam.__version__ if hasattr(pyoccam, '__version__') else '3.4.0'}")
if PANDAS_AVAILABLE:
    print("✅ Pandas available for enhanced table display")
print()

🧮 OCCAM Reconstructability Analysis System
Version: 4.2.0
✅ Pandas available for enhanced table display



## 📂 Load Data

In [3]:
# Check if data file exists
if not Path(DATA_FILE).exists():
    print(f"❌ ERROR: Data file '{DATA_FILE}' not found!")
    print(f"\nPlease ensure the file exists in the current directory:")
    print(f"   {Path.cwd()}")
    print(f"\nOr update DATA_FILE with the full path to your data.")
    raise FileNotFoundError(f"Data file not found: {DATA_FILE}")

# Initialize OCCAM manager
manager = pyoccam.VBMManager()

print(f"📁 Loading data: {DATA_FILE}")
if not manager.init_from_command_line(["occam", DATA_FILE]):
    raise RuntimeError("Failed to load data file. Check format (tab/comma delimited with header).")

# Use HTML format for beautiful display in notebooks!
manager.set_report_separator(pyoccam.HTMLFORMAT)

# Configure report columns
manager.set_report_variables("ID$I, Model, Level$I, h, ddf, dLR, Alpha, Inf, %dH(DV), dAIC, dBIC")
manager.set_ref_model("bottom")

# Set confusion matrix target if specified
if TARGET_STATE:
    manager.set_fit_classifier_target(TARGET_STATE)

# Display data summary
print(f"✅ Data loaded successfully!\n")
print(f"📊 Data Summary:")
print(f"   • Sample size: {manager.get_sample_size()}")
print(f"   • Variables: {', '.join(manager.get_variable_list())}")
print(f"   • DV (dependent variable): {manager.get_variable_list()[-1]}")
print(f"\n📋 Configuration:")
print(f"   • Search: {SEARCH_TYPE}, levels={SEARCH_LEVELS}, width={SEARCH_WIDTH}")
print(f"   • Selection: {SELECTION_CRITERION}")
if TARGET_STATE:
    print(f"   • Confusion matrix target: {TARGET_STATE}")

📁 Loading data: dementia05.txt
✅ Data loaded successfully!

📊 Data Summary:
   • Sample size: 424
   • Variables: APOE, Gender, Education, AgeLastExam, rs1801133, rs3818361, rs7561528, rs744373, rs6943822, rs4298437, rs7012010, rs11136000, rs10786998, rs11193130, rs610932, rs3851179, rs3764650, rs3865444, CaseControl
   • DV (dependent variable): CaseControl

📋 Configuration:
   • Search: loopless-up, levels=3, width=3
   • Selection: BIC
   • Confusion matrix target: 0


## 🔍 Search for Optimal Models

In [4]:
print("\n" + "="*60)
print("🔍 SEARCH PHASE")
print("="*60)
print(f"Searching for optimal models...")
print(f"(This may take {SEARCH_LEVELS * SEARCH_WIDTH} to {SEARCH_LEVELS * SEARCH_WIDTH * 5} seconds)\n")

# Run the search
start_time = time.time()
search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=False
)
search_time = time.time() - start_time

print(f"✅ Search completed in {search_time:.2f} seconds\n")

# Get best models
best_models = {
    "BIC": manager.get_best_model_by_bic(),
    "AIC": manager.get_best_model_by_aic(),
    "Information": manager.get_best_model_by_information()
}

# Display best models summary
print("🏆 Best Models Found:")
for criterion, model in best_models.items():
    if model:
        marker = "  →" if criterion == SELECTION_CRITERION else "   "
        print(f"{marker} {criterion:12} {model}")
    else:
        print(f"    {criterion:12} (not found)")

# Display detailed search results if requested
if SHOW_SEARCH_DETAILS:
    display_occam_html(search_report, "Search Results - All Models Evaluated")


🔍 SEARCH PHASE
Searching for optimal models...
(This may take 9 to 45 seconds)

✅ Search completed in 0.06 seconds

🏆 Best Models Found:
  → BIC          IV:ApZ
    AIC          IV:ApSxAZ
    Information  IV:ApSxAZ


### 📊 Search Results as Interactive Table

In [5]:
# Parse search results into pandas DataFrame for nice display
if PANDAS_AVAILABLE:
    try:
        # Switch to space format temporarily to parse the table
        manager.set_report_separator(pyoccam.SPACESEP)
        text_report = manager.generate_search_report(
            search_type=SEARCH_TYPE,
            levels=SEARCH_LEVELS,
            width=SEARCH_WIDTH,
            include_test_data=False
        )
        manager.set_report_separator(pyoccam.HTMLFORMAT)  # Switch back
        
        # Parse the text report
        lines = text_report.split('\n')
        
        # Find the header line (contains 'Model')
        header_line = None
        data_lines = []
        in_data = False
        
        for line in lines:
            if 'Model' in line and 'Level' in line:
                header_line = line
                in_data = True
                continue
            if in_data and line.strip():
                # Check if this looks like a data line (starts with a number)
                parts = line.split()
                if parts and parts[0].strip().isdigit():
                    data_lines.append(line)
                elif '=' in line or 'Search' in line:
                    break  # End of data section
        
        if header_line and data_lines:
            # Simple parsing - split by whitespace
            headers = header_line.split()
            
            # Parse data
            data = []
            for line in data_lines:
                parts = line.split()
                if len(parts) >= 3:  # At minimum: ID, Model, Level
                    data.append(parts)
            
            # Create DataFrame
            df = pd.DataFrame(data)
            
            # Set column names based on header count
            if len(df.columns) >= len(headers):
                df.columns = headers[:len(df.columns)]
            
            # Convert numeric columns
            numeric_cols = ['ID', 'Level', 'h', 'ddf', 'dLR', 'Alpha', 'Inf', '%dH(DV)', 'dAIC', 'dBIC']
            for col in numeric_cols:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            
            print(f'📊 Search Results Table ({len(df)} models evaluated):\n')
            
            # Show top models by each criterion
            if 'dBIC' in df.columns:
                print('Top 5 models by BIC (smaller is better):')
                display(df.nsmallest(5, 'dBIC', keep='first')[['Model', 'Level', 'dBIC', 'dAIC']].style.highlight_min(subset=['dBIC'], color='lightgreen'))
                print()
            
            if 'dAIC' in df.columns:
                print('Top 5 models by AIC (smaller is better):')
                display(df.nsmallest(5, 'dAIC', keep='first')[['Model', 'Level', 'dAIC', 'dBIC']].style.highlight_min(subset=['dAIC'], color='lightgreen'))
                print()
            
            if 'Inf' in df.columns:
                print('Top 5 models by Information (larger is better):')
                display(df.nlargest(5, 'Inf', keep='first')[['Model', 'Level', 'Inf', 'Alpha']].style.highlight_max(subset=['Inf'], color='lightgreen'))
            
            # Store for later use
            search_results_df = df
            
        else:
            print('Note: Could not parse search results into table format')
            print('The HTML display above shows the complete results.')
            search_results_df = None
            
    except Exception as e:
        print(f'Note: Could not create DataFrame table: {e}')
        print('The HTML display above shows the complete results.')
        search_results_df = None
else:
    print('Note: Install pandas for interactive table display: pip install pandas')
    search_results_df = None

📊 Search Results Table (5 models evaluated):



## 📈 Fit Analysis of Best Model

In [ ]:
# Get the selected model
print("Starting fit analysis...")
selected_model = best_models.get(SELECTION_CRITERION)
fit_report = None

if not selected_model or selected_model == "":
    print(f"⚠️ No model found using {SELECTION_CRITERION} criterion.")
    print("This can happen if:")
    print("  • The search didn't find any improving models")
    print("  • The data has limited structure")
    print("  • Try increasing SEARCH_LEVELS or SEARCH_WIDTH")
    print("\nYou can manually specify a model to fit:")
    print("  selected_model = 'IV:ABC'  # Replace with your model")
else:
    print("\n" + "="*60)
    print("📈 FIT ANALYSIS")
    print("="*60)
    print(f"Model: {selected_model}")
    print(f"Selected by: {SELECTION_CRITERION} criterion\n")
    
    print("Generating detailed fit analysis...")
    
    try:
        # Use the same approach that worked before - pass "0" directly
        # The confusion matrix target is already set globally via set_fit_classifier_target
        fit_report = manager.generate_fit_report(selected_model, TARGET_STATE)
        
        if fit_report:
            # Check what's included
            components = []
            if "Residuals" in fit_report:
                components.append("Residuals")
            if "Conditional DV" in fit_report or "Conditional probability" in fit_report:
                components.append("Conditional Probabilities")
            if "Confusion Matrix" in fit_report or "Confusion Matrices" in fit_report:
                components.append("Confusion Matrix")
            
            print(f"✅ Fit report generated with: {', '.join(components) if components else 'Basic statistics'}\n")
            
            # Display the fit report
            if SHOW_FIT_DETAILS:
                display_occam_html(fit_report, f"Fit Analysis: {selected_model}")
            
            # Extract and display key metrics if confusion matrix is present
            if "Confusion Matrix" in fit_report and TARGET_STATE:
                print("\n📊 Classification Performance:")
                
                # Try to extract accuracy from the report
                lines = fit_report.split('\n')
                for line in lines:
                    if "Accuracy" in line:
                        print(f"   {line.strip()}")
                        break
        else:
            print("⚠️ Fit report is empty")
            
    except Exception as e:
        print(f"\n⚠️ Error generating fit report: {e}")
        print("\nTroubleshooting:")
        print("  1. Check that the model name is valid")
        print("  2. Try a simpler model")
        print("  3. Check the C++ bindings")
        
        # Show model statistics from search as fallback
        if 'search_results_df' in globals() and search_results_df is not None:
            print("\nShowing model statistics from search:")
            model_stats = search_results_df[search_results_df['Model'] == selected_model]
            if not model_stats.empty:
                display(model_stats)

## 💾 Save Results to Files

In [ ]:
# Determine file extension based on format
ext_map = {"tab": "tsv", "comma": "csv", "space": "txt"}
file_ext = ext_map.get(FILE_FORMAT, "txt")

# Set appropriate separator for file output
separator_map = {
    "tab": pyoccam.TABSEP,
    "comma": pyoccam.COMMASEP,
    "space": pyoccam.SPACESEP
}

# Temporarily switch to text format for file saving
manager.set_report_separator(separator_map[FILE_FORMAT])

# Save search report
search_filename = f"search_{SEARCH_TYPE}_{SEARCH_LEVELS}levels.{file_ext}"
search_text = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=False
)
with open(search_filename, 'w') as f:
    f.write(search_text)
print(f"💾 Search report saved: {search_filename}")

# Save fit report if we have a model
if selected_model and selected_model != "":
    try:
        fit_filename = f"fit_{selected_model.replace(':', '_')}_{SELECTION_CRITERION}.{file_ext}"
        
        # Use the same TARGET_STATE as in the main fit analysis
        fit_text = manager.generate_fit_report(selected_model, TARGET_STATE)
        with open(fit_filename, 'w') as f:
            f.write(fit_text)
        print(f"💾 Fit report saved: {fit_filename}")
    except Exception as e:
        print(f"Note: Could not save fit report: {e}")
        fit_filename = None
else:
    fit_filename = None

# Save DataFrame to Excel if available
if 'search_results_df' in locals() and search_results_df is not None:
    try:
        excel_filename = f"search_{SEARCH_TYPE}_{SEARCH_LEVELS}levels.xlsx"
        search_results_df.to_excel(excel_filename, index=False)
        print(f"💾 Excel file saved: {excel_filename}")
    except:
        pass

# Switch back to HTML for any further notebook display
manager.set_report_separator(pyoccam.HTMLFORMAT)

print(f"\n✅ All results saved in {FILE_FORMAT} format!")

## 🔬 Advanced Analysis (Optional)

In [ ]:
# Function to analyze any specific model
def analyze_model(model_name):
    """Analyze a specific model by name (e.g., 'IV:ABC', 'IV:ApZ:EdZ')"""
    print(f"\nAnalyzing model: {model_name}")
    try:
        # Use the same approach as the main analysis - pass TARGET_STATE directly
        report = manager.generate_fit_report(model_name, TARGET_STATE)
        if report:
            display_occam_html(report, f"Custom Model Analysis: {model_name}")
        else:
            print("  ⚠️ Empty report generated")
    except Exception as e:
        print(f"  ❌ Error: {e}")

# Example: Analyze a specific model
# analyze_model("IV:ApZ:EdZ:CZ")  # Uncomment and modify to analyze your model

# Test with simplest model
# analyze_model("IV:Z")  # For testing

In [ ]:
# Compare all three best models side by side
def compare_best_models():
    """Generate fit reports for all three selection criteria"""
    print("\n" + "="*60)
    print("📊 COMPARISON OF BEST MODELS")
    print("="*60)
    
    for criterion, model in best_models.items():
        if model:
            print(f"\n{criterion}: {model}")
            try:
                report = manager.generate_fit_report(model, TARGET_STATE)
                display_occam_html(report, f"{criterion} Best Model: {model}")
            except Exception as e:
                print(f"  Error generating report: {e}")

# Uncomment to run comparison
# compare_best_models()

In [ ]:
# Interactive model exploration
def explore_models_at_level(level):
    """Show all models at a specific level"""
    if PANDAS_AVAILABLE and 'search_results_df' in globals() and search_results_df is not None:
        level_models = search_results_df[search_results_df['Level'] == level]
        if not level_models.empty:
            print(f"\nModels at level {level}:")
            display(level_models[['Model', 'dBIC', 'dAIC', 'Inf']])
        else:
            print(f"No models found at level {level}")
    else:
        print(f"\nModels at level {level}:")
        print("See the search report above for models at each level.")

# Custom search with different parameters
def custom_search(search_type, levels, width):
    """Run a custom search with different parameters"""
    print(f"\nRunning custom search: {search_type}, levels={levels}, width={width}")
    report = manager.generate_search_report(search_type, levels, width, False)
    display_occam_html(report, f"Custom Search: {search_type}")
    
# Example: Run a more thorough search
# custom_search("loopless-up", 5, 5)  # Uncomment to run

# Example: Explore models at specific level
# explore_models_at_level(3)  # Uncomment to see all level 3 models

## 📚 Summary and Next Steps

### What we've completed:
1. ✅ Loaded and validated the data
2. ✅ Searched for optimal models using OCCAM's algorithms
3. ✅ Selected the best model based on your chosen criterion
4. ✅ Generated detailed fit analysis with statistics
5. ✅ Saved results to files for further analysis

### Next steps you might consider:
- **Try different search parameters** to explore the model space more thoroughly
- **Compare models** using different selection criteria (BIC vs AIC vs Information)
- **Analyze specific models** of interest using the `analyze_model()` function
- **Use test data** for validation (set `include_test_data=True` in search)
- **Export results** to other tools for visualization or further analysis

### Understanding the output:
- **BIC/AIC**: Lower values indicate better models (penalized for complexity)
- **Information**: Higher values indicate more information captured
- **Alpha**: Significance level of the model
- **%correct**: Classification accuracy (if applicable)

### Need help?
- OCCAM documentation: https://occam-ra.github.io/
- Model notation: `IV:ABC` means variables A, B, C predict the DV
- Search types: loopless (no loops), disjoint (no overlaps), chain (linear), full (exhaustive)

In [ ]:
# Final summary
print("\n" + "="*60)
print("🎉 ANALYSIS COMPLETE!")
print("="*60)
print(f"\n📋 Final Summary:")
print(f"   • Data: {DATA_FILE} ({manager.get_sample_size()} samples)")
print(f"   • Search: {SEARCH_TYPE} ({search_time:.2f}s)")
if selected_model:
    print(f"   • Best {SELECTION_CRITERION} model: {selected_model}")
else:
    print(f"   • Best {SELECTION_CRITERION} model: Not found")
    
print(f"\n📂 Files Generated:")
print(f"   • Search report: {search_filename}")
if 'fit_filename' in locals() and fit_filename:
    print(f"   • Fit report: {fit_filename}")
if 'excel_filename' in locals():
    print(f"   • Excel file: {excel_filename}")
    
print(f"\nThank you for using OCCAM!")